# Forests

The `Forest` class is how NeuRosetta handles multiple neurons at the same time. 


In [1]:
import neurosetta as nr

forest = nr.load_example_data()

forest.summary()


ID,nodes,branches,leaves,cable,units,isReduced,Flag
720575940596125868,2010,99,111,404.2,micron,False,False
720575940599459782,1408,90,99,298.0,micron,False,False
720575940599704006,2043,140,156,429.4,micron,False,False
720575940599729862,2299,149,166,494.6,micron,False,False
TOTAL,7760,478,532,1626.3,micron,,
MEAN,1940,119.5,133,406.6,micron,,


## Basics

Forests have a number of fairly standard operations usable through `forest.<method()>`:

|Method|What it Does|
|-|-|
|`append`|Append a `Tree` to the last place|
|`insert`|Insert a `Tree` at index|
|`extend`|Append multiple `Tree`s from an iterable|
|`remove`|Remove a specific `Tree` passed to the method|
|`pop`|Remove and return the `Tree` at index|
|`clear`|Remove all `Tree`s|
|`ids`|Returns `Tree` ids in `Forest` order|
|`by_ids`|Return `Tree` or `Forest` subset by id(s)|
|`remove_id`|Remove `Tree`(s) from `Forest` by id(s)|




## Metadata

The `Forest` class has the same metadata summary tools as `Tree`, so you can again list the available metadata keys within the forest for all neurons:

In [2]:
forest.list_meta()

['Hemisphere', 'Neuron_subtype', 'Neuron_type']

You can additionally get a summary, telling you how many neurons in your forest have each of these keys:

In [3]:
forest.meta_summary()

{'Hemisphere': 4, 'Neuron_subtype': 4, 'Neuron_type': 4}

You can set new metadata entries for all neurons in your `Forest` using `forest.set_meta()` exactly like the `Tree` class:

In [4]:
forest.set_meta(key = 'test', value = 'still a test')
forest.list_meta()

['Hemisphere', 'Neuron_subtype', 'Neuron_type', 'test']

And deletion:

In [5]:
forest.del_meta(key = 'test')
forest.list_meta()

['Hemisphere', 'Neuron_subtype', 'Neuron_type']

## Indexing

Forests can be indexed, giving you a subset of the forest, using a neurons index in the forest, slicing, or a list of indices:

In [6]:
forest[0]

Tree(ID=720575940596125868) with 2010 nodes

In [7]:
forest[2:]

Forest(n=2, ids=[720575940599704006, 720575940599729862])

In [8]:
forest[[1,3]]

Forest(n=2, ids=[720575940599459782, 720575940599729862])

You can also use specific `Tree.ID` values using `forest.by_id`:

In [9]:
forest.by_id(forest.ids()[2:])

Forest(n=2, ids=[720575940599704006, 720575940599729862])

And get the indices of neurons within the `Forest` which have specific metadata values, and use these to index your `Forest`:

In [10]:
inds = forest.meta_indices(key = 'Neuron_subtype', value = 'a')
forest[inds]

Forest(n=2, ids=[720575940599704006, 720575940599729862])

This is nice and all but their is a *significantly* better way.

## Filtering

Using the filter method you can filter to specific elements of your neurons in a number of way.

First, and key in `Tree.metadata` can be used directly, so for example to get a `Forest` of neurons where `Neuron_subtype == 'a` within the metadata:

In [11]:
T5a = forest.filter(Neuron_subtype="a")
print(T5a.ids())

[720575940599704006, 720575940599729862]


Or with a list of options in the metadata:

In [12]:
T5ab = forest.filter(Neuron_subtype=["a","b"])
print(T5ab.ids())


[720575940599459782, 720575940599704006, 720575940599729862]


You can also use custom lambda functions combined with `tree.get_meta` to achieve the same results:

In [13]:
T5a = forest.filter(lambda t: t.get_meta("Neuron_subtype") == "a")
print(T5a.ids())

T5ab = forest.filter(lambda t: (t.get_meta("Neuron_subtype") == "a") | (t.get_meta("Neuron_subtype") == "b"))
print(T5ab.ids())

[720575940599704006, 720575940599729862]
[720575940599459782, 720575940599704006, 720575940599729862]


The use of lambda functions also extends to arbitary properties from your neurons, for example filtering to only those with a total cable length between $300$ and $450$:

In [14]:
min_c, max_c = 300.0, 450.0 
mid_cable = forest.filter(lambda t: min_c < t.get_total_cable_length() < max_c)
print(mid_cable.ids())

[720575940596125868, 720575940599704006]


More complicated operations can result in complicated lambdas though, so finally you can also specify your own custyom filter function, as long as it adears to a couple of rule:

- The finction must return a bool value.
- Any additional arguments in the function must use default values as they cannot be passed to filter.

We can reproduce the lambda value above like so:

In [15]:
def cable_in_range(tree, lo=min_c, hi=max_c):
    cable = tree.get_total_cable_length()
    return lo < cable < hi

mid_cable_fn = forest.filter(cable_in_range)
print(mid_cable_fn.ids())

[720575940596125868, 720575940599704006]


## Metrics

NeuRosetta can use (most) tree based metrics across an entire forest, a simple example being:


In [16]:
print(forest.count_nodes())
print(forest.get_total_cable_length())


[2010, 1408, 2043, 2299]
[404.2379751328887, 298.01680254283934, 429.44516470059705, 494.587506702061]


Like the `Tree` class, have a look at {doc}`../reference/metrics` where we try to keep an up to date list of implemented metrics for forests.

When using a `Forest` method, you have a couple of additional arguments:

| Argument | dtype | What it does |
|---|---|---|
| `parallel` | bool | Whether or not to use multithreading |
| `max_workers` | int | Max number of threads to use |
| `show_progress` | bool | Whether or not to show a progress bar |
| `merge_axis` | int | Which axis to merge the output along |
| `global_` | bool | If to apply the function globally, or individually to each `Tree` |
| `bind` | bool | Whether or not to bind the output to the `Tree` |

> **Note on `merge_axis`:** If the function would normally return a single value,
> such as `count_nodes`, this will raise a `TypeError`. Additionally, if the
> output is an array and you try to merge along an axis that you can't,
> you will get an `AxisError`.

> **Note on `global_`:** This is exclusively for functions, such as rotating
> all coordinates, which make sense to be applied to all neurons in the
> forest. The majority of functions don't have this, and you will get a
> `ValueError` if you try.

> **Note on `bind`:** `bind` will always be offered, but unless the underlying
> function being used has a `bind` argument, it will do nothing.

NeuRosetta has default values set for things like `parallel`,`max_workers`, and `show_progress`. You can access and change these in the `configure` module, and have a look at {doc}`../reference/configuration`.

An example of the `global_` argument can be seen with the `forest.get_convex_hull_volume()`. With `global_ = False` you get the volume of a convex hull fit to each neuron individually:

In [17]:
forest.get_convex_hull_volume()

[7397.91694898965, 3296.657501256841, 6854.223609505026, 6020.542831060736]

But if you use `global_ = True` you will get the volume of a convex hull fit around all points within the *whole forest*:

In [18]:
forest.get_convex_hull_volume(global_ = True)

86577.7971375561

A good example of `merge_axis` can be seen when getting node coordinates using `forest.get_root_coordinate`. If you leave `merge_axis = None` (the default behaviour) you will get a list of  1 x 3 numpy arrays, one for each neuron: 

In [19]:
forest.get_root_coordinate(merge_axis = None)

[array([723.45567568, 274.77648649, 242.55340541]),
 array([693.735, 325.521, 223.776]),
 array([707.10647368, 326.01298026, 226.48638158]),
 array([705.264, 327.558, 223.041])]

If you specify `merge_axis = 0` though, the output will be merge along the `0` axis into a single array:

In [20]:
forest.get_root_coordinate(merge_axis = 0)

array([[723.45567568, 274.77648649, 242.55340541],
       [693.735     , 325.521     , 223.776     ],
       [707.10647368, 326.01298026, 226.48638158],
       [705.264     , 327.558     , 223.041     ]])

## Custom `apply`

Although the single metric functionality is nice and all, NeuRosetta also has `forest.apply` functionality. This is actually what is going on under the hood with the above, so the same function arguments apply (except `global_`).

This means that this:

In [21]:
forest.count_nodes()

[2010, 1408, 2043, 2299]

and this:

In [22]:
forest.apply(fn = nr.count_nodes)

[2010, 1408, 2043, 2299]

are identical. 

> **Important:** when using `forest.apply` with a function that takes additional arguments, you should *explicitly use the argument keyword*, otherwise you will get strange outputs or errors.

`forest.apply` can be passed any function that takes a `Tree` object as input. Moreover, this means you can create your own custom functions. This is usefull for two reasons.

First, built in function methods if chained together will start a thread pool each time, you are better off wrapping the tree functions and then passing this to apply.

For example, if you want the total cable length and number of nodes in all your neurons it is quicker to create a single function. This is a fairly redundant example, but look at the timing differences bellow for the wrapped function example:


In [26]:
%%timeit
def process_one(tree):
    return tree.count_nodes(), tree.get_total_cable_length()

forest.apply(process_one)


300 μs ± 16 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


and for two repeated calls:

In [24]:
%%timeit
forest.count_nodes()
forest.get_total_cable_length()

487 μs ± 23.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Admitedly, in this example most of the overhead is taken up by setting up the parallelisation as the opperation itself is stupidly fast...

Anyway. The use of custom function passed to apply in this way also means you can develope complex analyses pipelines with your custom function. For example you could write a function which aligns each neuron so that it's eigen-axis are alligned with the global coordinate basis vectors, scale each neuron by some factor along its axis, and then get the convex hull volume of the result:

In [28]:
def process_one(tree):
    tree.align_coordinates()
    tree.scale_coordinates_along_pca(10)
    return tree.get_convex_hull_volume()

forest.apply(process_one)

[7397916948.989647, 3296657501.256849, 6854223609.505022, 6020542831.060724]

This is a bit of a stupid example, but so what. 

Next: {doc}`plotting`.
